# Urban Step 5: Full Live Autonomous System (2-Model + FSM + Pure Pursuit)

This notebook runs real-time **Urban Autonomous Driving** on JetRacer, integrating:
1. **Model 1**: Conditioned Waypoint Trajectory Model (TensorRT / ONNX)
2. **Model 2**: Object Detector (`green_light`, `red_light`, `turn_left_sign`, `turn_right_sign`, `stop_sign`, `crosswalk`)
3. **FSM Planner**: Decision Machine enforcing stopping **BEFORE the Pedestrian Crosswalk** when Red Light or STOP sign is active!
4. **Pure Pursuit Controller**: Steering angle $\delta = \arctan(2 L \sin\alpha / L_d)$ trajectory tracking.

### 1. Setup Environment & Load Models

In [ ]:
import os
import sys
import ctypes
from pathlib import Path

# Pre-load CUDA libraries into RTLD_GLOBAL symbol table
cuda_lib64 = "/usr/local/cuda/lib64"
for lib_name in ["libcudart.so", "libcudnn.so", "libcuda.so"]:
    lib_path = os.path.join(cuda_lib64, lib_name)
    if os.path.exists(lib_path):
        try:
            ctypes.CDLL(lib_path, mode=ctypes.RTLD_GLOBAL)
        except Exception:
            pass

# Add parent directory to sys.path
parent_dir = Path.cwd().parent.parent
if str(parent_dir) not in sys.path:
    sys.path.append(str(parent_dir))

import onnxruntime as ort
try:
    from jetracer.urban.detector import UrbanObjectDetector
    from jetracer.urban.fsm import UrbanFSMPlanner
    from jetracer.urban.pure_pursuit import PurePursuitController
    from jetracer.urban.runner import UrbanAutonomousRunner
    from jetracer.utils import preprocess_onnx, bgr8_to_jpeg
except ImportError:
    from urban.detector import UrbanObjectDetector
    from urban.fsm import UrbanFSMPlanner
    from urban.pure_pursuit import PurePursuitController
    from urban.runner import UrbanAutonomousRunner
    from utils import preprocess_onnx, bgr8_to_jpeg

# Locate Lane Model file (.engine or .onnx)
engine_path = os.path.join(Path.cwd(), "urban_conditioned_lane_model.engine")
onnx_path   = os.path.join(Path.cwd(), "urban_conditioned_lane_model.onnx")
lane_model_path = engine_path if os.path.exists(engine_path) else onnx_path

print(f"[*] Loading Urban Lane Model: {lane_model_path}")
available_providers = ort.get_available_providers()
providers = []
if 'TensorrtExecutionProvider' in available_providers:
    providers.append('TensorrtExecutionProvider')
if 'CUDAExecutionProvider' in available_providers:
    providers.append('CUDAExecutionProvider')
providers.append('CPUExecutionProvider')

lane_session = ort.InferenceSession(lane_model_path, providers=providers)
print(f"[+] Lane Session Providers: {lane_session.get_providers()}")

# Locate Detector Model file
detector_path = os.path.join(Path.cwd(), "urban_detector.onnx")
detector = UrbanObjectDetector(detector_path, conf_threshold=0.45, providers=providers)


### 2. Initialize ROS Node, Hardware & Controllers

In [ ]:
import rospy
from sensor_msgs.msg import Image as ROSImage
from jetracer.nvidia_racecar import NvidiaRacecar

try:
    rospy.init_node('urban_live_notebook', anonymous=True, disable_signals=True)
    print("[+] ROS Node initialized successfully!")
except Exception as e:
    print(f"[*] ROS Node notice: {e}")

car = NvidiaRacecar()
fsm = UrbanFSMPlanner(stop_duration=3.0)
pure_pursuit = PurePursuitController(wheelbase=0.175, default_lookahead=0.45)
print("[+] JetRacer Hardware, FSM Planner, and Pure Pursuit Controller Ready.")


### 3. Interactive UI Dashboard & ROS Subscriber Setup

In [ ]:
import cv2
import base64
import ipywidgets
from IPython.display import display

# Unregister previous ROS subscriber if active
if 'ros_sub' in globals() and ros_sub is not None:
    try:
        ros_sub.unregister()
    except Exception:
        pass

# Widgets
route_cmd_widget = ipywidgets.ToggleButtons(options=['LEFT', 'STRAIGHT', 'RIGHT'], description='Route Cmd', value='STRAIGHT')
drive_state_btn  = ipywidgets.ToggleButtons(options=['STOP', 'RUN'], description='Drive State', value='STOP')
lookahead_slider = ipywidgets.FloatSlider(min=0.2, max=1.0, step=0.05, value=0.45, description='Lookahead Ld', layout=ipywidgets.Layout(width='340px'))
throttle_slider  = ipywidgets.FloatSlider(min=0.0, max=1.0, step=0.01, value=0.25, description='Throttle', layout=ipywidgets.Layout(width='340px'))

camera_html_widget = ipywidgets.HTML(
    value="<p><b>Waiting for ROS Camera Topic...</b></p>",
    layout=ipywidgets.Layout(width='240px', height='240px')
)
status_html_widget = ipywidgets.HTML(value="<p>FSM Status: Initialized</p>")

# Callback to draw real-time predictions & detections on live camera feed
def on_urban_live_frame(cv_image, waypoints, detections, is_stopped, active_cmd, fsm_status, steering, dyn_throttle):
    h, w = cv_image.shape[:2]
    frame = cv_image.copy()
    
    # 1. Draw detected bounding boxes (Green Light, Red Light, Crosswalk, STOP sign, etc.)
    for d in detections:
        bbox = d['bbox']
        c_name = d['class_name']
        x1, y1, x2, y2 = int(bbox[0]*w), int(bbox[1]*h), int(bbox[2]*w), int(bbox[3]*h)
        
        color = (0, 255, 0) if c_name == 'green_light' else (0, 0, 255) if c_name in ['red_light', 'stop_sign'] else (255, 255, 0)
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        cv2.putText(frame, f"{c_name} {d['confidence']:.2f}", (x1, max(15, y1-5)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 2)
        
    # 2. Draw predicted 5 Waypoints Trajectory Spline
    pts = ((waypoints + 1.0) / 2.0 * np.array([w, h])).astype(int)
    for i in range(len(pts) - 1):
        cv2.line(frame, tuple(pts[i]), tuple(pts[i+1]), (0, 255, 255), 2)
    for pt in pts:
        cv2.circle(frame, tuple(pt), 5, (0, 255, 0), -1)
        
    # 3. Draw steering overlay text
    state_txt = "STOPPED" if is_stopped else "GO"
    cv2.putText(frame, f"[{active_cmd}] {state_txt} Steer:{steering:+.2f} Thr:{dyn_throttle:.2f}", (10, 25),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 0), 2)
                
    jpeg_bytes = bgr8_to_jpeg(frame)
    b64 = base64.b64encode(jpeg_bytes).decode('utf-8')
    
    html_str = f'''
    <div style="font-family: monospace; background: #1e1e1e; color: #00ff00; padding: 8px; border-radius: 8px; display: inline-block;">
        <h5 style="margin:0 0 4px 0; color: #ffffff;">Urban Autonomous Dashboard</h5>
        <img src="data:image/jpeg;base64,{b64}" style="width:224px; height:224px; border:2px solid #00ff00; border-radius:4px;" />
    </div>
    '''
    camera_html_widget.value = html_str
    
    color_style = "color: red;" if is_stopped else "color: green;"
    status_html_widget.value = f'''
    <div style="font-family: monospace; background: #252525; padding: 10px; border-radius: 6px;">
        <p style="margin: 2px 0; {color_style}"><b>FSM State:</b> {fsm.state}</p>
        <p style="margin: 2px 0; color: #ffffff;"><b>Active Route:</b> {active_cmd}</p>
        <p style="margin: 2px 0; color: #00ffff;"><b>Notice:</b> {fsm_status}</p>
    </div>
    '''

# Setup Urban Runner
runner = UrbanAutonomousRunner(
    lane_session=lane_session,
    detector=detector,
    fsm=fsm,
    controller=pure_pursuit,
    car=car,
    route_command=lambda: route_cmd_widget.value,
    lookahead=lambda: lookahead_slider.value,
    throttle=lambda: throttle_slider.value,
    on_frame=on_urban_live_frame
)
runner.running = False

def on_state_change(change):
    if change['new'] == 'RUN':
        runner.running = True
        print("[+] Urban Autonomous Driving ACTIVE!")
    else:
        runner.stop()
        print("[*] Urban Autonomous Driving STOPPED.")

drive_state_btn.observe(on_state_change, names='value')

topic_name = "/csi_cam_0/image_raw"
ros_sub = rospy.Subscriber(topic_name, ROSImage, runner.image_callback, queue_size=1, buff_size=2**24)

# UI Layout
controls_box = ipywidgets.VBox([
    drive_state_btn,
    route_cmd_widget,
    lookahead_slider,
    throttle_slider,
    status_html_widget
])

dashboard = ipywidgets.HBox([camera_html_widget, controls_box])
display(dashboard)
print(f"[*] Subscribed to ROS Camera Topic: {topic_name}")


### 4. Emergency Stop Cell

In [ ]:
# Emergency Stop Cell
drive_state_btn.value = 'STOP'
runner.stop()
